# S11 - BI/ML distribuido con Spark

**Proyecto Sello — Equipo LLSW3.** Dos mecanismos de publicación hacia el mismo Pushgateway, Prometheus y Grafana:

- **Parte A — BI batch (una sola vez):** KPIs del flujo de eventos calculados sobre el Parquet de observabilidad de **S9** (`s09_red_observabilidad`, telemetría **real** de las NIC de esta PC, topic `red-telemetria`), publicados una vez con `PUT`.
- **Parte B — ML en vivo (continuo):** el `PipelineModel` de **S10** (`modelo_ce_lr`) recargado **sin reentrenar**, aplicado sobre `campo-electrico-eventos` y publicado en el Pushgateway en **cada micro-lote**.

```text
Parte A (una vez)                              Parte B (continuo)
Parquet S9 s09_red_observabilidad              Kafka campo-electrico-eventos (1 lectura/s)
   └─► KPI: volumen, validez, latencia            └─► PipelineModel de S10 (recargado)
          │ PUT, una vez                                └─► foreachBatch: Parquet propio + PUT por micro-lote
          └──────────────► Pushgateway :9091 ◄──────────────┘
                               └─► Prometheus :9090 (honor_labels) ─► Grafana "S11 - BI/ML distribuido"
```

## 3.1 Entorno y Pushgateway

`obs/compose.yml` suma un tercer servicio, `pushgateway` (`prom/pushgateway:v1.11.1`, puerto host 49091), y `obs/prometheus/prometheus.yml` un *scrape job* con `honor_labels: true` para que Prometheus respete los `job`/`instance` que trae cada *push*. Se verifica desde el notebook que los tres *targets* están `UP`:

In [1]:
import json, time, datetime, urllib.request, urllib.parse
LIMA = datetime.timezone(datetime.timedelta(hours=-5))
print("Ejecutado:", datetime.datetime.now(LIMA).strftime("%Y-%m-%d %H:%M:%S"), "(Lima)")
def prom(path):
    with urllib.request.urlopen("http://prometheus:9090/api/v1" + path, timeout=10) as r:
        return json.loads(r.read())["data"]
for t in prom("/targets")["activeTargets"]:
    print(f"  {t['labels']['job']:<15} {t['scrapeUrl']:<38} {t['health'].upper()}")

Ejecutado: 2026-10-27 15:18:53 (Lima)
  kafka-exporter  http://kafka-exporter:9308/metrics     UP
  prometheus      http://localhost:9090/metrics          UP
  pushgateway     http://pushgateway:9091/metrics        UP


## Parte A — BI batch

### 3.2 `SparkSession` y Parquet real de S9

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark = (
    SparkSession.builder
    .appName("sesion11-bi-ml-distribuido")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "America/Lima")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-24766d37-6605-40d9-b5a9-8ae04850cbff;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central


	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 in central
	found org.apache.commons#commons-pool2;2.13.1 in central
:: resolution report :: resolve 240ms :: artifacts dl 10ms
	:: modules in use:
	at.yawk.lz4#lz4-java;1.11.0 from central in [default]
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	org.apache.commons#commons-pool2;2.13.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.5.0 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.5.0 from central in [default]
	org.apache.kafka#kafka-clients;3.9.2 from central in [default]
	org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 from central 

26/10/27 20:18:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 4.2.0


In [3]:
RUTA_EVENTOS_S9 = "/opt/artifacts/s09_red_observabilidad"
eventos = spark.read.parquet(RUTA_EVENTOS_S9)
print(f"Filas eventos (S9): {eventos.count():,}")
eventos.select("sensorId", "partition", "offset", "isValid", "processedAt", "latencyMs", "transporteMs").orderBy("offset").show(5, truncate=False)
print("Rango de la evidencia:", eventos.agg(F.min("kafkaTimestamp"), F.max("kafkaTimestamp")).first())

Filas eventos (S9): 1,090


+----------+---------+------+-------+-------------+---------+------------+
|sensorId  |partition|offset|isValid|processedAt  |latencyMs|transporteMs|
+----------+---------+------+-------+-------------+---------+------------+
|Ethernet 2|0        |15020 |true   |1792019615004|2968     |-526        |
|Ethernet 2|0        |15021 |true   |1792019615004|1942     |-572        |
|Ethernet 2|0        |15022 |true   |1792019620005|5916     |965         |
|Ethernet 2|0        |15023 |true   |1792019620005|4889     |919         |
|Ethernet 2|0        |15024 |true   |1792019620005|3860     |868         |
+----------+---------+------+-------+-------------+---------+------------+
only showing top 5 rows


Rango de la evidencia: Row(min(kafkaTimestamp)=datetime.datetime(2026, 10, 14, 23, 13, 31, 509000), max(kafkaTimestamp)=datetime.datetime(2026, 10, 14, 23, 19, 44, 234000))


Es la evidencia que S9 dejó persistida el 14/10: lecturas reales de 3 NIC durante ~6 minutos, con `isValid`, `processedAt` y `latencyMs` ya calculados en Spark. No se recalcula nada del streaming de S9.

### 3.3 KPI del flujo de eventos

In [4]:
kpi_eventos = eventos.agg(
    F.count("*").alias("total"),
    F.avg(F.col("isValid").cast("int")).alias("ratioValidos"),
    F.avg("latencyMs").alias("avgLatencyMs"),
    F.percentile_approx("latencyMs", 0.95).alias("p95LatencyMs"),
).first()
print(f"Eventos procesados:   {kpi_eventos['total']}")
print(f"Ratio de válidos:     {kpi_eventos['ratioValidos']:.4f}")
print(f"Latencia promedio:    {kpi_eventos['avgLatencyMs']:.2f} ms")
print(f"Latencia p95:         {kpi_eventos['p95LatencyMs']} ms")
print("\nDetalle por NIC (KPI de negocio por dispositivo):")
eventos.groupBy("sensorId").agg(F.count("*").alias("eventos"), F.round(F.avg(F.col("isValid").cast("int")), 4).alias("ratioValidos"),
                                F.round(F.avg("latencyMs"), 1).alias("avgLatencyMs")).orderBy("sensorId").show(truncate=False)

Eventos procesados:   1090
Ratio de válidos:     0.9991
Latencia promedio:    2701.76 ms
Latencia p95:         5053 ms

Detalle por NIC (KPI de negocio por dispositivo):


+----------------------------------+-------+------------+------------+
|sensorId                          |eventos|ratioValidos|avgLatencyMs|
+----------------------------------+-------+------------+------------+
|Ethernet                          |364    |0.9973      |2698.4      |
|Ethernet 2                        |363    |1.0         |2696.5      |
|vEthernet (WSL (Hyper-V firewall))|363    |1.0         |2710.3      |
+----------------------------------+-------+------------+------------+



**Lectura de negocio:** el 99,9 % de las lecturas cumplió el contrato `red.lectura` (la única inválida es la lectura con tasa negativa que S9 inyectó a propósito). La latencia promedio (~2,7 s) queda **por debajo** del umbral sensible de 4 200 ms que S9 propuso con estos mismos datos, así que la alerta no se habría disparado: la latencia está dominada por la espera al trigger de 5 s, no por el transporte.

### 3.4 Publicar el KPI una sola vez (`PUT`) y verificarlo en Prometheus

In [5]:
metricas_flujo = {
    "lambda26_bi_eventos_total": kpi_eventos["total"],
    "lambda26_bi_eventos_validos_ratio": kpi_eventos["ratioValidos"],
    "lambda26_bi_latencia_promedio_ms": kpi_eventos["avgLatencyMs"],
    "lambda26_bi_latencia_p95_ms": kpi_eventos["p95LatencyMs"],
}
cuerpo = "".join(f"# TYPE {n} gauge\n{n} {v}\n" for n, v in metricas_flujo.items())
print(cuerpo)
PUSHGATEWAY_FLUJO_URL = "http://pushgateway:9091/metrics/job/lambda26_bi_flujo/instance/s11_batch"
peticion = urllib.request.Request(PUSHGATEWAY_FLUJO_URL, data=cuerpo.encode("utf-8"), method="PUT")
with urllib.request.urlopen(peticion) as respuesta:
    print("Pushgateway respondió:", respuesta.status)

# TYPE lambda26_bi_eventos_total gauge
lambda26_bi_eventos_total 1090
# TYPE lambda26_bi_eventos_validos_ratio gauge
lambda26_bi_eventos_validos_ratio 0.9990825688073395
# TYPE lambda26_bi_latencia_promedio_ms gauge
lambda26_bi_latencia_promedio_ms 2701.75871559633
# TYPE lambda26_bi_latencia_p95_ms gauge
lambda26_bi_latencia_p95_ms 5053

Pushgateway respondió: 200


In [6]:
def consultar(expr):
    r = prom("/query?query=" + urllib.parse.quote(expr))["result"]
    return r[0]["value"] if r else None

print("Inmediatamente después del PUT:")
for n in metricas_flujo:
    print(f"  {n:<36} {consultar(n) or 'SIN DATOS (aún no hubo scrape)'}")
time.sleep(20)   # al menos un scrape_interval (15 s)
print("\n20 s después (Prometheus ya hizo scrape al Pushgateway):")
for n in metricas_flujo:
    ts, v = consultar(n)
    print(f"  {n:<36} {v}   (scrape {datetime.datetime.fromtimestamp(ts, LIMA).strftime('%H:%M:%S')})")

Inmediatamente después del PUT:
  lambda26_bi_eventos_total            [1793132352.951, '1090']
  lambda26_bi_eventos_validos_ratio    [1793132352.952, '0.9990825688073395']
  lambda26_bi_latencia_promedio_ms     [1793132352.953, '2701.75871559633']
  lambda26_bi_latencia_p95_ms          [1793132352.954, '5053']



20 s después (Prometheus ya hizo scrape al Pushgateway):
  lambda26_bi_eventos_total            1090   (scrape 15:19:32)
  lambda26_bi_eventos_validos_ratio    0.9990825688073395   (scrape 15:19:32)
  lambda26_bi_latencia_promedio_ms     2701.75871559633   (scrape 15:19:32)
  lambda26_bi_latencia_p95_ms          5053   (scrape 15:19:32)


## Parte B — ML en vivo

### 3.5 Productor de `campo-electrico-eventos`

El productor de S10 se vuelve a levantar sin reconstruir nada (`docker compose -f uso-campo-electrico/compose.yml up -d` y `docker exec ... python producer_campo_electrico.py`). Publica una lectura por segundo con el mismo simulador con el que se generó el histórico de S04. Se verifica que el topic recibe lecturas nuevas:

In [7]:
lote = (spark.read.format("kafka").option("kafka.bootstrap.servers", "kafka:9092")
        .option("subscribe", "campo-electrico-eventos").option("startingOffsets", "earliest").load())
ultimos = lote.select("offset", F.date_format("timestamp", "yyyy-MM-dd HH:mm:ss").alias("llegada")).orderBy(F.desc("offset")).limit(3)
print("mensajes en el topic:", lote.count())
ultimos.show(truncate=False)

mensajes en el topic: 1869


+------+-------------------+
|offset|llegada            |
+------+-------------------+
|1869  |2026-10-27 15:19:33|
|1868  |2026-10-27 15:19:32|
|1867  |2026-10-27 15:19:31|
+------+-------------------+



### 3.6 Cargar el modelo que S10 ya entrenó (sin reentrenar)

In [8]:
from pyspark.ml import PipelineModel
RUTA_MODELO_S10 = "/opt/artifacts/s10/models/modelo_ce_lr"
modelo_cargado = PipelineModel.load(RUTA_MODELO_S10)
lr = modelo_cargado.stages[-1]
print("Modelo cargado desde:", RUTA_MODELO_S10)
print("Etapas:", [type(s).__name__ for s in modelo_cargado.stages])
print("Columnas de entrada:", modelo_cargado.stages[0].getInputCols())
print("Coeficiente de CE:", round(float(lr.coefficients[0]), 5), "| intercepto:", round(lr.intercept, 4), "(los mismos de S10)")

Modelo cargado desde: /opt/artifacts/s10/models/modelo_ce_lr
Etapas: ['VectorAssembler', 'LinearRegressionModel']
Columnas de entrada: ['CE', 'CM', 'TempOut', 'OutHum', 'WindSpeed', 'Bar', 'SolarRad', 'UVIndex']
Coeficiente de CE: 0.95092 | intercepto: -0.2441 (los mismos de S10)


### 3.7 Leer el stream y aplicar el modelo, sin ventanas ni estado

In [9]:
from pyspark.sql.types import StructType, StringType, DoubleType, LongType
esquema_evento = (
    StructType().add("tipoEvento", StringType()).add("estacionId", StringType())
    .add("CE", DoubleType()).add("CM", DoubleType()).add("TempOut", DoubleType()).add("OutHum", DoubleType())
    .add("WindSpeed", DoubleType()).add("Bar", DoubleType()).add("SolarRad", DoubleType()).add("UVIndex", DoubleType())
    .add("origen", StringType()).add("timestamp", LongType())
)
crudo = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "campo-electrico-eventos")
    .option("startingOffsets", "latest")
    .option("failOnDataLoss", "false")
    .load()
)
eventos_stream = crudo.select(F.from_json(F.col("value").cast("string"), esquema_evento).alias("e")).select("e.*")
predicciones_stream = modelo_cargado.transform(eventos_stream)
print("¿streaming?", predicciones_stream.isStreaming)

¿streaming? True


### 3.8 Publicar cada predicción en el Pushgateway, micro-lote por micro-lote

Cada micro-lote se guarda en el Parquet propio de S11 **y** se publica con `PUT` bajo `job=lambda26_ml_streaming`, `instance=<estacionId>`. Además del par (real, predicción) se publica el **error absoluto en vivo**: la predicción que el modelo hizo en la lectura anterior comparada con el `CE` real que acaba de llegar. Es la comparación "datos del día siguiente contra datos en vivo" de la Regla #37, hecha mientras ocurre.

In [10]:
import os, shutil
RUTA_PREDICCIONES_STREAM = "/opt/artifacts/s11/predicciones_stream"
CHECKPOINT_STREAM = "/opt/artifacts/s11/chk_predicciones_stream"
shutil.rmtree(RUTA_PREDICCIONES_STREAM, ignore_errors=True); shutil.rmtree(CHECKPOINT_STREAM, ignore_errors=True)
PUSHGATEWAY_BASE = "http://pushgateway:9091/metrics/job/lambda26_ml_streaming/instance"
ultima_prediccion = {}   # en el driver: la predicción pendiente de cada estación

def publicar_prediccion_en_vivo(estacion_id, ce_real, prediccion, error_abs, id_lote):
    cuerpo = (
        "# TYPE lambda26_ml_ce_real gauge\n" f"lambda26_ml_ce_real {ce_real}\n"
        "# TYPE lambda26_ml_prediccion_siguiente gauge\n" f"lambda26_ml_prediccion_siguiente {prediccion}\n"
        "# TYPE lambda26_ml_batch_id gauge\n" f"lambda26_ml_batch_id {id_lote}\n"
    )
    if error_abs is not None:
        cuerpo += "# TYPE lambda26_ml_error_absoluto gauge\n" f"lambda26_ml_error_absoluto {error_abs}\n"
    peticion = urllib.request.Request(f"{PUSHGATEWAY_BASE}/{estacion_id}", data=cuerpo.encode("utf-8"), method="PUT")
    urllib.request.urlopen(peticion).close()

def guardar_predicciones_microlote(df_microlote, id_lote):
    filas = df_microlote.count()
    if filas == 0:
        print(f"Micro-lote {id_lote}: sin filas")
        return
    salida = df_microlote.select(
        F.lit(id_lote).alias("batchId"), "estacionId",
        (F.col("timestamp") / 1000).cast("timestamp").alias("fechaHora"), "CE",
        F.col("prediction").alias("prediccionSiguienteCE"),
    )
    salida.write.mode("append").parquet(RUTA_PREDICCIONES_STREAM)
    for fila in salida.orderBy("fechaHora").select("estacionId", F.date_format("fechaHora", "HH:mm:ss").alias("hora"), "CE", "prediccionSiguienteCE").collect():
        anterior = ultima_prediccion.get(fila["estacionId"])
        error_abs = abs(anterior - fila["CE"]) if anterior is not None else None
        publicar_prediccion_en_vivo(fila["estacionId"], fila["CE"], fila["prediccionSiguienteCE"], error_abs, id_lote)
        ultima_prediccion[fila["estacionId"]] = fila["prediccionSiguienteCE"]
    err = f"{error_abs:.4f}" if error_abs is not None else "-"
    print(f"Micro-lote {id_lote}: {filas} lectura(s) guardadas y publicadas | {fila['hora']} CE={fila['CE']:.2f} pred={fila['prediccionSiguienteCE']:.4f} |error|={err}")

# Momento 1: arrancar la consulta
consulta_predicciones = (
    predicciones_stream.writeStream
    .foreachBatch(guardar_predicciones_microlote)
    .option("checkpointLocation", CHECKPOINT_STREAM)
    .trigger(processingTime="1 second")
    .start()
)
time.sleep(25)

Micro-lote 0: sin filas


netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory


Micro-lote 1: 1 lectura(s) guardadas y publicadas | 15:19:37 CE=-2.23 pred=-2.2211 |error|=-


Micro-lote 2: 3 lectura(s) guardadas y publicadas | 15:19:40 CE=-2.17 pred=-2.1622 |error|=0.0415


Micro-lote 3: 2 lectura(s) guardadas y publicadas | 15:19:42 CE=-2.25 pred=-2.2388 |error|=0.0402


Micro-lote 4: 2 lectura(s) guardadas y publicadas | 15:19:44 CE=-2.17 pred=-2.1626 |error|=0.0596


Micro-lote 5: 2 lectura(s) guardadas y publicadas | 15:19:46 CE=-2.19 pred=-2.1834 |error|=0.0232


Micro-lote 6: 2 lectura(s) guardadas y publicadas | 15:19:49 CE=-2.12 pred=-2.1170 |error|=0.0031


Micro-lote 7: 3 lectura(s) guardadas y publicadas | 15:19:52 CE=-2.10 pred=-2.0954 |error|=0.0585


Micro-lote 8: 3 lectura(s) guardadas y publicadas | 15:19:55 CE=-2.08 pred=-2.0757 |error|=0.0359


### 3.9 Confirmar que las métricas de verdad cambian en Prometheus

In [11]:
def leer(nombre):
    v = consultar(nombre)
    return (datetime.datetime.fromtimestamp(v[0], LIMA).strftime("%H:%M:%S"), v[1]) if v else None

primera = leer("lambda26_ml_ce_real")
print("Primera lectura:            ", primera)
time.sleep(20)
segunda = leer("lambda26_ml_ce_real")
print("Segunda lectura (20 s después):", segunda)
print("¿Cambió el valor?", primera is not None and segunda is not None and primera[1] != segunda[1])

print("\nSerie que Prometheus guardó durante el último minuto (scrape cada 15 s):")
rango = prom("/query?query=" + urllib.parse.quote("lambda26_ml_ce_real[1m]"))["result"][0]["values"]
for ts, v in rango:
    print(f"  {datetime.datetime.fromtimestamp(ts, LIMA).strftime('%H:%M:%S')}  CE real = {v}")

Primera lectura:             ('15:20:02', '-2.19')
Micro-lote 9: 4 lectura(s) guardadas y publicadas | 15:19:59 CE=-2.15 pred=-2.1421 |error|=0.0475


Micro-lote 10: 3 lectura(s) guardadas y publicadas | 15:20:02 CE=-2.03 pred=-2.0285 |error|=0.0832


Micro-lote 11: 2 lectura(s) guardadas y publicadas | 15:20:04 CE=-2.14 pred=-2.1357 |error|=0.0634


Micro-lote 12: 2 lectura(s) guardadas y publicadas | 15:20:06 CE=-2.19 pred=-2.1836 |error|=0.0456


Micro-lote 13: 2 lectura(s) guardadas y publicadas | 15:20:08 CE=-2.25 pred=-2.2409 |error|=0.0370


...

Micro-lote 15: 2 lectura(s) guardadas y publicadas | 15:20:12 CE=-2.13 pred=-2.1261 |error|=0.0544


Micro-lote 16: 2 lectura(s) guardadas y publicadas | 15:20:14 CE=-2.07 pred=-2.0695 |error|=0.0096


Micro-lote 17: 2 lectura(s) guardadas y publicadas | 15:20:16 CE=-1.98 pred=-1.9827 |error|=0.0793


Micro-lote 18: 2 lectura(s) guardadas y publicadas | 15:20:18 CE=-2.01 pred=-2.0081 |error|=0.0385


Segunda lectura (20 s después): ('15:20:22', '-1.98')
¿Cambió el valor? True

Serie que Prometheus guardó durante el último minuto (scrape cada 15 s):
  15:19:50  CE real = -2.19
  15:20:05  CE real = -2.03
  15:20:20  CE real = -1.98


Dejamos el stream corriendo unos minutos más para que el tablero de Grafana acumule la línea en vivo (las capturas del tablero se toman durante esta celda).

In [12]:
for i in range(6):
    time.sleep(30)
    p = consulta_predicciones.lastProgress
    print(f"{datetime.datetime.now(LIMA).strftime('%H:%M:%S')} | batchId={p['batchId']} | "
          f"CE real={leer('lambda26_ml_ce_real')[1]} | predicción={leer('lambda26_ml_prediccion_siguiente')[1]} | "
          f"|error|={leer('lambda26_ml_error_absoluto')[1]}")

Micro-lote 19: 2 lectura(s) guardadas y publicadas | 15:20:20 CE=-2.04 pred=-2.0393 |error|=0.0572


Micro-lote 20: 2 lectura(s) guardadas y publicadas | 15:20:22 CE=-2.02 pred=-2.0173 |error|=0.0750


Micro-lote 21: 2 lectura(s) guardadas y publicadas | 15:20:24 CE=-2.00 pred=-1.9963 |error|=0.0338


Micro-lote 22: 2 lectura(s) guardadas y publicadas | 15:20:26 CE=-2.03 pred=-2.0245 |error|=0.0524


Micro-lote 23: 2 lectura(s) guardadas y publicadas | 15:20:28 CE=-2.13 pred=-2.1187 |error|=0.0674


...

Micro-lote 97: 3 lectura(s) guardadas y publicadas | 15:23:09 CE=-2.15 pred=-2.1626 |error|=0.0109


Micro-lote 98: 4 lectura(s) guardadas y publicadas | 15:23:13 CE=-2.15 pred=-2.1649 |error|=0.0595


Micro-lote 99: 3 lectura(s) guardadas y publicadas | 15:23:16 CE=-2.17 pred=-2.1826 |error|=0.0063


Micro-lote 100: 2 lectura(s) guardadas y publicadas | 15:23:18 CE=-2.27 pred=-2.2777 |error|=0.0104


15:23:22 | batchId=100 | CE real=-2.17 | predicción=-2.1825603560411264 | |error|=0.006282082073591155


### 3.10 Detener la consulta y auditar lo recolectado (Regla #37)

In [13]:
# Momento 2: detener la consulta
consulta_predicciones.stop()
print("consulta activa:", consulta_predicciones.isActive)

consulta activa: False


Exception in thread "stream execution thread for [id = 87fbc6f0-848c-417a-b338-0849b87dc9ae, runId = 0ff7e28c-7e6c-40fa-ac57-d164d2eba017]" java.lang.StackOverflowError
	[... 1024 líneas de traza Java omitidas ...]


**Hallazgo: el valor queda congelado en el Pushgateway.** Con la consulta detenida, se consulta dos veces la misma métrica separada por más de un *scrape*:

In [14]:
time.sleep(20)   # un scrape completo para que Prometheus tenga el último valor publicado antes del stop
a = leer("lambda26_ml_ce_real"); time.sleep(30); b = leer("lambda26_ml_ce_real")
print("Después del stop:", a, "->", b, "(30 s y dos scrapes después)")
print("Mismo valor (congelado), aunque Prometheus lo volvió a scrapear:", a[1] == b[1])
pg = urllib.request.urlopen("http://pushgateway:9091/api/v1/metrics").read().decode()
grupo = [g for g in json.loads(pg)["data"] if g["labels"].get("job") == "lambda26_ml_streaming"][0]
print("Último push registrado en el Pushgateway:", grupo["push_time_seconds"]["metrics"][0]["value"] if "push_time_seconds" in grupo else grupo["last_push_successful"])
print("up del pushgateway sigue en:", consultar('up{job="pushgateway"}')[1], "— no hay señal de que el job murió")

Después del stop: ('15:23:42', '-2.27') -> ('15:24:12', '-2.27') (30 s y dos scrapes después)
Mismo valor (congelado), aunque Prometheus lo volvió a scrapear: True
Último push registrado en el Pushgateway: 1.7931326008606365e+09
up del pushgateway sigue en: 1 — no hay señal de que el job murió


In [15]:
predicciones_hoy = spark.read.parquet(RUTA_PREDICCIONES_STREAM)
print("predicciones guardadas:", predicciones_hoy.count(), "| micro-lotes:", predicciones_hoy.select("batchId").distinct().count())
print("lecturas por micro-lote:")
predicciones_hoy.groupBy("batchId").count().groupBy(F.col("count").alias("lecturasEnElLote")).count().orderBy("lecturasEnElLote").show()

# Método de la guía: unir el micro-lote N con el N+1
actual_siguiente = predicciones_hoy.select((F.col("batchId") - 1).alias("batchId"), "estacionId", F.col("CE").alias("CE_real_siguiente"))
comparado_guia = (predicciones_hoy.alias("p").join(actual_siguiente.alias("a"), ["batchId", "estacionId"])
                  .withColumn("errorAbsoluto", F.abs(F.col("prediccionSiguienteCE") - F.col("CE_real_siguiente"))))
g = comparado_guia.agg(F.count("*").alias("n"), F.avg("errorAbsoluto").alias("mae")).first()
print(f"Método batchId N vs N+1: {g['n']} pares (más que predicciones si hay lotes con 2+ lecturas) | MAE={g['mae']:.4f}")

predicciones guardadas: 223 | micro-lotes: 101
lecturas por micro-lote:


+----------------+-----+
|lecturasEnElLote|count|
+----------------+-----+
|               1|    1|
|               2|   83|
|               3|   13|
|               4|    3|
|               5|    1|
+----------------+-----+



Método batchId N vs N+1: 497 pares (más que predicciones si hay lotes con 2+ lecturas) | MAE=0.0573


In [16]:
from pyspark.sql.window import Window
w = Window.partitionBy("estacionId").orderBy("fechaHora")
comparado = (
    predicciones_hoy.withColumn("CE_real_siguiente", F.lead("CE", 1).over(w))
    .na.drop(subset=["CE_real_siguiente"])
    .withColumn("errorAbsoluto", F.abs(F.col("prediccionSiguienteCE") - F.col("CE_real_siguiente")))
    .withColumn("errorPersistencia", F.abs(F.col("CE") - F.col("CE_real_siguiente")))
)
comparado.select("batchId", F.date_format("fechaHora", "HH:mm:ss").alias("hora"), "CE", "prediccionSiguienteCE", "CE_real_siguiente", "errorAbsoluto") \
    .orderBy("fechaHora").show(8, truncate=False)
kpi_prediccion = comparado.agg(
    F.count("*").alias("totalComparado"), F.avg("errorAbsoluto").alias("maeProduccion"),
    F.avg("errorPersistencia").alias("maePersistencia"),
).first()
print(f"Predicciones comparables: {kpi_prediccion['totalComparado']}")
print(f"MAE en producción:        {kpi_prediccion['maeProduccion']:.4f}   (S10 offline: 0.0401)")
print(f"MAE de la persistencia:   {kpi_prediccion['maePersistencia']:.4f}   (S10 offline: 0.0405)")

+-------+--------+-----+---------------------+-----------------+---------------------+
|batchId|hora    |CE   |prediccionSiguienteCE|CE_real_siguiente|errorAbsoluto        |
+-------+--------+-----+---------------------+-----------------+---------------------+
|1      |15:19:37|-2.23|-2.221059717278858   |-2.28            |0.05894028272114182  |
|2      |15:19:38|-2.28|-2.2692564442889083  |-2.22            |0.049256444288908074 |
|2      |15:19:39|-2.22|-2.2115438929317714  |-2.17            |0.04154389293177152  |
|2      |15:19:40|-2.17|-2.162246238065201   |-2.22            |0.0577537619347992   |
|3      |15:19:41|-2.22|-2.2097836216588678  |-2.25            |0.04021637834113223  |
|3      |15:19:42|-2.25|-2.2387937527435993  |-2.24            |0.0012062472564009497|
|4      |15:19:43|-2.24|-2.2295836291516857  |-2.17            |0.0595836291516858   |
|4      |15:19:44|-2.17|-2.1626487250225472  |-2.22            |0.05735127497745296  |
+-------+--------+-----+-------------------

Predicciones comparables: 222
MAE en producción:        0.0407   (S10 offline: 0.0401)
MAE de la persistencia:   0.0411   (S10 offline: 0.0405)


Limpieza del valor congelado: se borra el grupo del Pushgateway y se confirma que Prometheus deja de devolver la serie (marcada como *stale* en el siguiente *scrape*).

In [17]:
for est in ultima_prediccion:
    urllib.request.urlopen(urllib.request.Request(f"{PUSHGATEWAY_BASE}/{est}", method="DELETE")).close()
    print("Grupo borrado del Pushgateway:", est)
time.sleep(20)
print("lambda26_ml_ce_real ahora:", consultar("lambda26_ml_ce_real") or "SIN DATOS (serie cortada, señal honesta de que el job no publica)")
print("KPI batch sigue disponible:", consultar("lambda26_bi_eventos_validos_ratio")[1])

Grupo borrado del Pushgateway: estacion-ns-01


lambda26_ml_ce_real ahora: SIN DATOS (serie cortada, señal honesta de que el job no publica)
KPI batch sigue disponible: 0.9990825688073395


### 3.11 Tablero de Grafana

El tablero **S11 - BI/ML distribuido** se provisiona desde `obs/grafana/dashboards/s11-bi-ml.json`: una fila con los 4 *Stat* del KPI batch (eventos, ratio de válidos en porcentaje, latencia promedio y p95), una fila con el *Time series* en vivo de `lambda26_ml_ce_real` y `lambda26_ml_prediccion_siguiente` (más el error absoluto) con rango *Last 5 minutes* y *refresh* 5 s, y una fila con la infraestructura de S9.

### 3.12 Cierre de la Unidad II

El estado del modelo ya no se mira abriendo el notebook de S10: está en el mismo Grafana que la infraestructura, junto al KPI del flujo de eventos. S12 integra ingesta, streaming, observabilidad y esta capa de BI/ML en el producto de la unidad.